# Fig. 2E:
## Variation partitioning of DNA viral community composition

In [14]:
#  Produces six files:
#
#    outputs/tables/Table_varpart_fractions_Bact_Abundance.csv
#    outputs/tables/Table_varpart_fractions_Bact_Activity.csv
#    outputs/figures/Fig_varpart_venn_Bact_Abundance.pdf
#    outputs/figures/Fig_varpart_venn_Bact_Activity.pdf
#    outputs/figures/Fig_varpart4_bar_Detail_Bact_Abundance.pdf   -> Fig. 2E
#    outputs/figures/Fig_varpart4_bar_Detail_Bact_Activity.pdf    -> Supp. Fig. 3
#
#  TWO PARTITIONINGS
#
#      3-block   Infauna (meio+macro) | Bacteria (PCo1-2) | Fluxes (O2+CH4)
#                -> how much is shared        (fraction table + Venn)
#      4-block   Meiofauna | Macrofauna | Fluxes | Bacteria
#                -> which block holds the unique variation, with meiofauna and
#                   macrofauna separated so they are no longer confounded
#                   (bar chart + partial db-RDA p-values)
#
#  Each runs twice, once per bacterial proxy (metagenome abundance,
#  metatranscriptome activity).
#  All of it is relative abundance - Hellinger - Bray-Curtis.

# 1. SETUP

PROJ <- "path/"
PROJ <- normalizePath(PROJ, mustWork = TRUE)
setwd(PROJ)

dir.create(file.path(PROJ, "outputs", "figures"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(PROJ, "outputs", "tables"),  recursive = TRUE, showWarnings = FALSE)

# 2. CONFIGURATION
# Both proxies are listed here and run in one pass, so they cannot drift apart.

VIRUS  <- "data/DNA_Viruses119.txt"
MASTER <- "data/Master_table_short.txt"

PROXY <- c(Bact_Abundance = "data/Bacteria_abundance_metagenome.txt",
           Bact_Activity  = "data/Abundance_Transcript_bacteria.txt")

PREDICTORS <- c("Meiofauna", "Macrofauna", "O2_Cons", "CH4_Flux",
                "BactPCo1", "BactPCo2")
 
SAMPLE_RE <- "^(Unmanip|CTRL|HMM|HM|LM)[ _]"

PREDICTORS <- c("Meiofauna", "Macrofauna", "O2_Cons", "CH4_Flux",
                "BactPCo1", "BactPCo2")

N_PERM <- 999
SEED   <- 1


# Fraction of the 4-block partitioning plotted as the last bar.
SHARED_FRACTION <- "[m]"

# Blocks with unique variance at or below this (in %) are dropped from the bar
# chart. Adjusted R2 can be negative, which is not plottable; macrofauna falls
# below the line in both runs and is therefore absent from both figures.
MIN_PCT <- 0.3

# 3. VIRAL COMMUNITY
# Relative abundance, then Hellinger. Hellinger is the square root of relative
# abundance

dv     <- read.table(VIRUS, header = TRUE, sep = "\t", row.names = 1, check.names = FALSE)
vh_all <- decostand(decostand(t(as.matrix(dv)), "total"), "hellinger")
dim(vh_all)   # samples x genomes

# 4. BACTERIAL COMMUNITY -> 2 AXES
# The bacterial table has 1,966-3,043 genera, which cannot enter a partitioning
# of 26 samples. The community is reduced to its first two PCoA axes, computed
# on the full matrix with the same relative-abundance -> Hellinger ->
# Bray-Curtis chain used for the viruses. Two axes keep the dominant
# compositional gradients while costing only 2 degrees of freedom — matching
# the other two blocks (2 each), so no block is privileged by size.

bact_axes <- function(path, k = 2) {
  b <- read.table(path, header = TRUE, sep = "\t", check.names = FALSE,
                  quote = "", fill = TRUE)

  # first column holds the taxon name; make.unique because genus names repeat
  if ("Genus" %in% colnames(b)) {
    rownames(b) <- make.unique(as.character(b$Genus)); b$Genus <- NULL
  } else {
    rownames(b) <- make.unique(as.character(b[[1]])); b[[1]] <- NULL
  }

  # keeping only sample columns (named by treatment); drop any trailing annotation
  scol <- grep(SAMPLE_RE, colnames(b), value = TRUE)
  if (!length(scol))
    stop("No sample columns matched SAMPLE_RE in ", path,
         ". Columns are: ", paste(head(colnames(b), 10), collapse = ", "))
  mat  <- as.matrix(sapply(b[, scol], function(x) as.numeric(as.character(x))))
  rownames(mat) <- rownames(b)
  colnames(mat) <- gsub(" ", "_", scol)   # harmonise with the master table
  mat[is.na(mat)] <- 0

  pco <- cmdscale(vegdist(decostand(decostand(t(mat), "total"), "hellinger"), "bray"), k = k)
  colnames(pco) <- paste0("BactPCo", seq_len(k))
  as.data.frame(pco)
}

# 5. PREDICTORS / SAMPLE ALIGNMENT
# Predictors are standardised last, putting blocks of different units
# (ind. m-2 vs mmol m-2 d-1) on a common scale. This happens after the
# drops, so the scaling reflects the cores actually analysed.

build_design <- function(bact_path) {
  bact <- bact_axes(bact_path, k = 2)

  env <- read.table(MASTER, header = TRUE, sep = "\t", check.names = FALSE,
                    na.strings = c("Na", "NA", ""), quote = "", comment.char = "")
  colnames(env) <- c("Sample", "Group", "CH4_Conc", "CH4_Flux",
                     "O2_Cons", "Macrofauna", "Meiofauna")

  # report before merging, while both sides are still visible
  no_bact <- setdiff(rownames(vh_all), rownames(bact))
  no_env  <- setdiff(rownames(vh_all), as.character(env$Sample))
  if (length(no_env))
    cat("  (a) not in master table   :", paste(no_env,  collapse = ", "), "\n")

  env <- merge(env, data.frame(Sample = rownames(bact), bact), by = "Sample")

  idx     <- match(rownames(vh_all), env$Sample)
  m       <- env[idx, , drop = FALSE]
  matched <- !is.na(idx)

  # alignment check, on the rows that matched, unmatched rows are legitimate
  stopifnot(identical(as.character(m$Sample[matched]), rownames(vh_all)[matched]))

  full <- matched & complete.cases(m[, PREDICTORS])
  lost <- rownames(vh_all)[matched & !complete.cases(m[, PREDICTORS])]
  if (length(lost))
    cat("  (b) missing a predictor   :", paste(lost, collapse = ", "), "\n")

  vh <- vh_all[full, , drop = FALSE]
  m  <- m[full, , drop = FALSE]

  for (v in PREDICTORS) m[[v]] <- as.numeric(scale(m[[v]]))

  cat("  cores analysed            :", sum(full), "of", nrow(vh_all), "\n")
  list(vh = vh, m = m, n = sum(full), used = rownames(vh_all)[full])
}

# 6. THREE BLOCKS, HOW MUCH IS SHARED
# Writing Table_varpart_fractions_<tag>.csv and Fig_varpart_venn_<tag>.pdf

varpart3_run <- function(d, tag) {
  Xinf <- d$m[, c("Meiofauna", "Macrofauna")]
  Xbac <- d$m[, c("BactPCo1", "BactPCo2")]
  Xche <- d$m[, c("O2_Cons", "CH4_Flux")]

  vp <- varpart(d$vh, Xinf, Xbac, Xche)
  print(vp)

  write.csv(data.frame(fraction = rownames(vp$part$indfract), vp$part$indfract),
            file.path("outputs", "tables", paste0("Table_varpart_fractions_", tag, ".csv")),
            row.names = FALSE)

  pdf(file.path("outputs", "figures", paste0("Fig_varpart_venn_", tag, ".pdf")),
      width = 7.5, height = 6.5)
  plot(vp, digits = 2,
       Xnames = c("Infauna", "Bacteria", "Chemistry"),
       bg = c("#E9C46A", "#457B9D", "#2A9D8F"), cex = 1.1)
  title(main = "Variation partitioning of DNA viral community composition")
  dev.off()

  invisible(vp)
}

# 7. FOUR BLOCKS: WHICH BLOCK IS UNIQUE
# varpart returns adjusted R2 but no p-values, so the significance printed on
# the figure comes from partial db-RDA, each block tested while conditioning
# on all the others, not from the partitioning itself.

varpart4_run <- function(d, tag) {
  m  <- d$m          # named locally so the formulas below resolve
  vh <- d$vh

  vp4 <- varpart(vh, ~Meiofauna, ~Macrofauna, ~O2_Cons + CH4_Flux,
                 ~BactPCo1 + BactPCo2, data = m)
  print(vp4)
  fr <- vp4$part$indfract

  write.csv(data.frame(fraction = rownames(fr),
                       adj_R2   = round(fr$Adj.R.square, 4),
                       pct      = round(100 * fr$Adj.R.square, 2)),
            file.path("outputs", "tables", paste0("Table_varpart4_fractions_", tag, ".csv")),
            row.names = FALSE)

  # unique fractions are the first four rows, in the order the blocks were passed
  uniq <- setNames(fr$Adj.R.square[1:4],
                   c("Meiofauna", "Macrofauna", "Chemistry", "Bacteria"))

  shared <- fr$Adj.R.square[rownames(fr) == SHARED_FRACTION]
  if (length(shared) != 1 || is.na(shared))
    stop("Fraction ", SHARED_FRACTION, " not found in the varpart table. Available: ",
         paste(rownames(fr), collapse = ", "))

  # significance of each unique fraction (partial db-RDA)
  blocks <- c(Meiofauna  = "Meiofauna",
              Macrofauna = "Macrofauna",
              Chemistry  = "O2_Cons + CH4_Flux",
              Bacteria   = "BactPCo1 + BactPCo2")

  pval <- sapply(names(blocks), function(nm) {
    cond <- paste(blocks[setdiff(names(blocks), nm)], collapse = " + ")
    set.seed(SEED)
    anova(dbrda(as.formula(paste0("vh ~ ", blocks[nm], " + Condition(", cond, ")")),
                data = m),
          permutations = N_PERM)$`Pr(>F)`[1]
  })

  dfp <- data.frame(
    Block = c("Bacteria\n(unique)", "Chemistry\n(unique)", "Meiofauna\n(unique)",
              "Macrofauna\n(unique)", "Shared\n(all)"),
    pct   = 100 * c(uniq["Bacteria"], uniq["Chemistry"], uniq["Meiofauna"],
                    uniq["Macrofauna"], shared),
    plab  = c(sprintf("p=%.3f", pval["Bacteria"]),
              sprintf("p=%.3f", pval["Chemistry"]),
              sprintf("p=%.3f", pval["Meiofauna"]),
              sprintf("p=%.3f", pval["Macrofauna"]), "-"),
    stringsAsFactors = FALSE
  )
  dfp <- dfp[dfp$pct > MIN_PCT, ]

  cols <- c("#457B9D", "#2A9D8F", "#E9C46A", "#6A4C93", "#B08968")[seq_len(nrow(dfp))]

  pdf(file.path("outputs", "figures", paste0("Fig_varpart4_bar_Detail_", tag, ".pdf")),
      width = 8, height = 5)
  par(mar = c(6, 4.5, 3, 1))
  bp <- barplot(dfp$pct, names.arg = dfp$Block, col = cols, border = "grey20", las = 1,
                ylab = "Unique variance explained (adj. R2, %)",
                ylim = c(0, max(dfp$pct) * 1.25),
                main = "Independent drivers of DNA viral community composition")
  text(bp, dfp$pct + max(dfp$pct) * 0.03,
       paste0(sprintf("%.1f%%", dfp$pct), "\n", dfp$plab),
       cex = 0.8, pos = 3, xpd = TRUE)
  dev.off()

  print(data.frame(Block = gsub("\n", " ", dfp$Block),
                   pct   = round(dfp$pct, 2),
                   p     = dfp$plab), row.names = FALSE)
  invisible(dfp)
}


# 8. RUN BOTH PROXIES

used <- list()

for (tag in names(PROXY)) {
  cat("\n\n##########  ", tag, "  ##########\n")
  d <- build_design(PROXY[[tag]])
  used[[tag]] <- d$used
  cat("\n--- 3 blocks ---\n")
  varpart3_run(d, tag)
  cat("\n--- 4 blocks ---\n")
  varpart4_run(d, tag)
}

cat("\n\n=== sample sets compared ===\n")
cat("activity only :", paste(setdiff(used$Bact_Activity, used$Bact_Abundance),
                             collapse = ", "), "\n")
cat("in both       :", length(intersect(used$Bact_Abundance, used$Bact_Activity)), "cores\n")

cat("\nDone. Six files written to outputs/tables/ and outputs/figures/\n")

sessionInfo()


[1]  26 119



##########   Bact_Abundance   ##########
  (b) missing a predictor   : HMM_6 
  cores analysed            : 25 of 26 

--- 3 blocks ---

Partition of variance in RDA 

Call: varpart(Y = d$vh, X = Xinf, Xbac, Xche)

Explanatory tables:
X1:  Xinf
X2:  Xbac
X3:  Xche 

No. of explanatory tables: 3 
Total variation (SS): 2.3506 
            Variance: 0.097943 
No. of observations: 25 

Partition table:
                      Df R.square Adj.R.square Testable
[a+d+f+g] = X1         2  0.16176      0.08555     TRUE
[b+d+e+g] = X2         2  0.24210      0.17320     TRUE
[c+e+f+g] = X3         2  0.16671      0.09096     TRUE
[a+b+d+e+f+g] = X1+X2  4  0.34206      0.21047     TRUE
[a+c+d+e+f+g] = X1+X3  4  0.29651      0.15582     TRUE
[b+c+d+e+f+g] = X2+X3  4  0.34593      0.21511     TRUE
[a+b+c+d+e+f+g] = All  6  0.45984      0.27978     TRUE
Individual fractions                                   
[a] = X1 | X2+X3       2               0.06467     TRUE
[b] = X2 | X1+X3       2            

R version 4.1.1 (2021-08-10)
Platform: x86_64-apple-darwin13.4.0 (64-bit)
Running under: macOS 26.4.1

Matrix products: default
BLAS/LAPACK: /Users/au657143/mambaforge/envs/maaslin/lib/libopenblasp-r0.3.23.dylib

locale:
[1] en_AU/en_AU/en_AU/C/en_AU/en_AU

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] vegan_2.6-4    lattice_0.21-8 permute_0.9-7 

loaded via a namespace (and not attached):
 [1] cluster_2.1.4   splines_4.1.1   MASS_7.3-60     uuid_1.2-0     
 [5] rlang_1.1.6     fastmap_1.1.1   tools_4.1.1     parallel_4.1.1 
 [9] grid_4.1.1      nlme_3.1-162    mgcv_1.9-0      cli_3.6.5      
[13] htmltools_0.5.7 digest_0.6.37   lifecycle_1.0.4 crayon_1.5.3   
[17] Matrix_1.6-0    IRdisplay_1.1   repr_1.1.6      base64enc_0.1-3
[21] vctrs_0.6.5     IRkernel_1.3.2  glue_1.8.0      evaluate_0.23  
[25] pbdZMQ_0.3-9    compiler_4.1.1  pillar_1.10.2   jsonlite_2.0.0 